# 1D spectral element prototype for LowLevelFEM

This notebook tests the smallest possible spectral-element layer on top of the existing LowLevelFEM/Gmsh machinery.

The idea is deliberately close to the existing P1/P2 transformation approach:

$$
\mathbf u_\mathrm{gmsh} = T\,\mathbf u_\mathrm{GLL},
$$

so that an already assembled finite-element matrix can be transformed as

$$
K_\mathrm{GLL} = T^\mathsf{T} K_\mathrm{gmsh} T,
\qquad
M_\mathrm{GLL} = T^\mathsf{T} M_\mathrm{gmsh} T.
$$

For this first experiment we use:

- one 1D high-order Gmsh element,
- the **actual local node coordinates reported by Gmsh**,
- Legendre–Gauss–Lobatto (GLL) nodes and weights generated algorithmically,
- a square interpolation transformation $T$,
- an independent direct GLL reference calculation,
- GLL nodal quadrature for the diagonal mass matrix.

No modification of LowLevelFEM internals is required in this notebook.

In [1]:
using LowLevelFEM
using LinearAlgebra
using SparseArrays
using Printf

p = 4          # polynomial order
L = 1.0        # element length
n = 3          # start with exactly one element

line_mesh(lx=L, n=n, order=p)

## 1. Read the high-order Gmsh element

A crucial point is that we do **not** assume a local ordering such as
`linspace(-1,1,p+1)`.

For example, Gmsh typically stores the two end nodes first and the internal
high-order nodes afterwards. Therefore the spectral transformation must use
the reference coordinates returned by `gmsh.model.mesh.getElementProperties`.

In [2]:
elemTypes, elemTags, elemNodeTags = gmsh.model.mesh.getElements(1, -1)

@assert length(elemTypes) == 1 "This prototype expects a single 1D element type."

etype = elemTypes[1]

elementName, elementDim, p_gmsh, numNodes, localNodeCoord, numPrimaryNodes =
    gmsh.model.mesh.getElementProperties(etype)

@assert p_gmsh == p
@assert length(localNodeCoord) == numNodes
@assert length(elemTags[1]) == n

ξ_gmsh = Float64.(localNodeCoord)

connectivity =
    permutedims(reshape(Int.(elemNodeTags[1]), numNodes, :))

nodeTags, nodeCoord, _ = gmsh.model.mesh.getNodes()
nodeTags = Int.(nodeTags)
XYZ = permutedims(reshape(Float64.(nodeCoord), 3, :))

tagToRow = Dict(tag => i for (i, tag) in enumerate(nodeTags))

# Physical coordinates of the first element in Gmsh local-node ordering.
X_gmsh = XYZ[[tagToRow[tag] for tag in connectivity[1, :]], :]

println("Element type : ", elementName)
println("Order        : ", p_gmsh)
println("Number nodes : ", numNodes)
println()
println("Gmsh local ordering:")
display(hcat(connectivity[1, :], ξ_gmsh, X_gmsh[:, 1]))

Element type : Line 5
Order        : 4
Number nodes : 5

Gmsh local ordering:


5×3 Matrix{Float64}:
 1.0  -1.0  0.0
 3.0   1.0  0.333333
 5.0  -0.5  0.0833333
 6.0   0.0  0.166667
 7.0   0.5  0.25

The displayed columns are:

1. global Gmsh node tag,
2. reference coordinate $\xi$,
3. physical $x$ coordinate.

The transformation below follows this Gmsh ordering exactly.

## 2. Legendre–Gauss–Lobatto nodes and weights

For polynomial order $p$, the GLL rule contains $p+1$ points:

- $\xi_0=-1$,
- $\xi_p=+1$,
- the $p-1$ interior roots of $P_p'(\xi)$.

The weights are

$$
w_i =
\frac{2}
{p(p+1)\,[P_p(\xi_i)]^2}.
$$

The implementation is general in $p$. For the first LowLevelFEM implementation
we can still decide to document/test only a conservative range such as
$1\le p\le10$.

In [3]:
"""
    legendre_pair(p, x)

Return `(Pp, Ppm1)`, where `Pp = P_p(x)` and `Ppm1 = P_{p-1}(x)`.
"""
function legendre_pair(p::Int, x::Real)
    p ≥ 0 || throw(ArgumentError("p must be non-negative."))

    p == 0 && return (1.0, 0.0)

    P0 = 1.0
    P1 = Float64(x)

    p == 1 && return (P1, P0)

    for k in 2:p
        Pk = ((2k - 1) * x * P1 - (k - 1) * P0) / k
        P0, P1 = P1, Pk
    end

    return P1, P0
end


"""
    gll(p; tol=1e-14, maxiter=100)

Compute the `p+1` Legendre–Gauss–Lobatto nodes and quadrature weights
on `[-1,1]`.
"""
function gll(p::Int; tol=1e-14, maxiter=100)
    p ≥ 1 || throw(ArgumentError("p must be at least one."))

    ξ = zeros(Float64, p + 1)
    ξ[1] = -1.0
    ξ[end] = 1.0

    # Chebyshev–Lobatto points are excellent Newton initial guesses.
    for k in 1:p-1
        x = -cos(pi * k / p)

        for _ in 1:maxiter
            Pp, Ppm1 = legendre_pair(p, x)

            dPp =
                p * (Ppm1 - x * Pp) / (1 - x^2)

            # Legendre differential equation:
            # (1-x^2) P'' - 2x P' + p(p+1)P = 0
            ddPp =
                (2x * dPp - p * (p + 1) * Pp) / (1 - x^2)

            Δx = dPp / ddPp
            x -= Δx

            abs(Δx) ≤ tol * max(1.0, abs(x)) && break
        end

        ξ[k + 1] = x
    end

    sort!(ξ)

    w = similar(ξ)
    for i in eachindex(ξ)
        Pp, _ = legendre_pair(p, ξ[i])
        w[i] = 2 / (p * (p + 1) * Pp^2)
    end

    return ξ, w
end


ξ_gll, w_gll = gll(p)

println("GLL nodes:")
display(ξ_gll)

println("GLL weights:")
display(w_gll)

@show sum(w_gll)

GLL nodes:


5-element Vector{Float64}:
 -1.0
 -0.6546536707079772
  0.0
  0.6546536707079772
  1.0

GLL weights:


5-element Vector{Float64}:
 0.1
 0.5444444444444444
 0.7111111111111111
 0.5444444444444444
 0.1

sum(w_gll) = 2.0


2.0

## 3. Lagrange interpolation and transformation matrices

Let $\mathbf u_G$ denote nodal values in the GLL basis and
$\mathbf u_E$ nodal values in the original Gmsh basis.

The matrix

$$
T_{ij}=L_j^\mathrm{GLL}(\xi_i^\mathrm{gmsh})
$$

gives

$$
\mathbf u_E=T\mathbf u_G.
$$

The reverse interpolation matrix is

$$
R_{ij}=L_j^\mathrm{gmsh}(\xi_i^\mathrm{GLL}),
$$

so for two bases spanning the same $P_p$ polynomial space,

$$
RT \approx I,\qquad TR\approx I.
$$

In [4]:
"""
    barycentric_weights(nodes)

Compute barycentric interpolation weights for distinct interpolation nodes.
"""
function barycentric_weights(nodes::AbstractVector)
    n = length(nodes)
    w = ones(Float64, n)

    for j in 1:n
        for k in 1:n
            j == k && continue
            w[j] /= nodes[j] - nodes[k]
        end
    end

    return w
end


"""
    lagrange_matrices(nodes, points; atol=1e-13)

Return `(N, D)`, where `N[q,j]` is the `j`th Lagrange polynomial evaluated
at `points[q]`, and `D[q,j]` is its derivative with respect to the reference
coordinate.
"""
function lagrange_matrices(
    nodes::AbstractVector,
    points::AbstractVector;
    atol=1e-13
)
    nodes = Float64.(nodes)
    points = Float64.(points)

    n = length(nodes)
    m = length(points)

    bw = barycentric_weights(nodes)

    N = zeros(Float64, m, n)
    D = zeros(Float64, m, n)

    for q in 1:m
        x = points[q]

        hit = findfirst(j -> isapprox(x, nodes[j]; atol=atol, rtol=0.0), 1:n)

        if hit !== nothing
            i = hit
            N[q, i] = 1.0

            for j in 1:n
                j == i && continue
                D[q, j] =
                    bw[j] / (bw[i] * (nodes[i] - nodes[j]))
            end

            D[q, i] = -sum(D[q, j] for j in 1:n if j != i)
        else
            dx = x .- nodes
            a = bw ./ dx
            S = sum(a)

            Nq = a ./ S
            N[q, :] .= Nq

            S2 = sum(bw ./ dx.^2)
            D[q, :] .= Nq .* (S2 / S .- 1.0 ./ dx)
        end
    end

    return N, D
end


T, _ = lagrange_matrices(ξ_gll, ξ_gmsh)
R, _ = lagrange_matrices(ξ_gmsh, ξ_gll)

Iref = Matrix{Float64}(I, numNodes, numNodes)

println("T : GLL DOFs -> Gmsh DOFs")
display(round.(T, digits=10))

println("R : Gmsh DOFs -> GLL DOFs")
display(round.(R, digits=10))

@show norm(R * T - Iref, Inf)
@show norm(T * R - Iref, Inf)

T : GLL DOFs -> Gmsh DOFs


5×5 Matrix{Float64}:
  1.0         0.0       0.0      0.0        0.0
  0.0         0.0       0.0      0.0        1.0
 -0.117188    0.884032  0.3125  -0.118407   0.0390625
  0.0         0.0       1.0      0.0        0.0
  0.0390625  -0.118407  0.3125   0.884032  -0.117188

R : Gmsh DOFs -> GLL DOFs


5×5 Matrix{Float64}:
  1.0         0.0        0.0        0.0       0.0
  0.128955   -0.0269146  1.15184   -0.408163  0.154277
  0.0         0.0        0.0        1.0       0.0
 -0.0269146   0.128955   0.154277  -0.408163  1.15184
  0.0         1.0        0.0        0.0       0.0

norm(R * T - Iref, Inf) = 3.3306690738754696e-16
norm(T * R - Iref, Inf) = 3.3306690738754696e-16


3.3306690738754696e-16

In [5]:
"""
    insert_unique!(A, i, j, value; atol=1e-12)

Insert an entry into a global transformation matrix represented by a
dictionary. If the entry already exists, verify that the duplicated
contribution is identical.

This is needed at shared element endpoints: adjacent elements define the
same transformation row there, which must not be added twice.
"""
function insert_unique!(
    A::Dict{Tuple{Int,Int},Float64},
    i::Int,
    j::Int,
    value::Real;
    atol=1e-12
)
    abs(value) < atol && return

    key = (i, j)
    value = Float64(value)

    if haskey(A, key)
        isapprox(A[key], value; atol=atol, rtol=atol) ||
            error(
                "Inconsistent transformation at global entry " *
                "($i,$j): $(A[key]) vs. $value"
            )
    else
        A[key] = value
    end

    return
end


"""
    sparse_from_dict(A, m, n)

Convert a dictionary of matrix entries into a sparse matrix.
"""
function sparse_from_dict(
    A::Dict{Tuple{Int,Int},Float64},
    m::Int,
    n::Int
)
    I = Int[]
    J = Int[]
    V = Float64[]

    sizehint!(I, length(A))
    sizehint!(J, length(A))
    sizehint!(V, length(A))

    for ((i, j), value) in A
        push!(I, i)
        push!(J, j)
        push!(V, value)
    end

    return sparse(I, J, V, m, n)
end


# ------------------------------------------------------------------
# Global spectral transformation
# ------------------------------------------------------------------

ndof = length(nodeTags)

# Gmsh node tag -> global matrix row/column
tagToDof = Dict(tag => i for (i, tag) in enumerate(nodeTags))

# GLL local ordering is monotonically increasing in ξ.
perm_gll = sortperm(ξ_gmsh)

Tentries = Dict{Tuple{Int,Int},Float64}()
Rentries = Dict{Tuple{Int,Int},Float64}()

for e in axes(connectivity, 1)

    # Gmsh local ordering
    gmsh_tags = vec(connectivity[e, :])

    # Same abstract global node tags, but ordered according to GLL ξ.
    gll_tags = gmsh_tags[perm_gll]

    gmsh_dofs = [tagToDof[tag] for tag in gmsh_tags]
    gll_dofs  = [tagToDof[tag] for tag in gll_tags]

    for i in 1:numNodes
        for j in 1:numNodes
            insert_unique!(
                Tentries,
                gmsh_dofs[i],
                gll_dofs[j],
                T[i, j]
            )

            insert_unique!(
                Rentries,
                gll_dofs[i],
                gmsh_dofs[j],
                R[i, j]
            )
        end
    end
end

T_global = sparse_from_dict(Tentries, ndof, ndof)
R_global = sparse_from_dict(Rentries, ndof, ndof)

I_global = spdiagm(0 => ones(ndof))

println("Global transformation:")
@show size(T_global)
@show nnz(T_global)
@show size(R_global)
@show nnz(R_global)

println()
println("Inverse checks:")
@show norm(Matrix(R_global * T_global - I_global), Inf)
@show norm(Matrix(T_global * R_global - I_global), Inf)

Global transformation:
size(T_global) = (13, 13)
nnz(T_global) = 37
size(R_global) = (13, 13)
nnz(R_global) = 37

Inverse checks:
norm(Matrix(R_global * T_global - I_global), Inf) = 4.440892098500626e-16
norm(Matrix(T_global * R_global - I_global), Inf) = 4.440892098500626e-16


4.440892098500626e-16

## 4. Physical coordinates of the GLL nodes

The Gmsh high-order geometry is retained. We only evaluate that geometry at
the GLL reference coordinates:

$$
\mathbf X_\mathrm{GLL}=R\,\mathbf X_\mathrm{gmsh}.
$$

Thus Gmsh remains responsible for the mesh topology, physical groups and
element connectivity; the spectral layer only introduces a different nodal
representation of the same polynomial geometry.

In [6]:
X_gll = R * X_gmsh

println("Reference and physical GLL coordinates:")
display(hcat(ξ_gll, X_gll[:, 1], X_gll[:, 2], X_gll[:, 3]))

Reference and physical GLL coordinates:


5×4 Matrix{Float64}:
 -1.0       0.0        0.0  0.0
 -0.654654  0.0575577  0.0  0.0
  0.0       0.166667   0.0  0.0
  0.654654  0.275776   0.0  0.0
  1.0       0.333333   0.0  0.0

For the present straight element, the GLL points are simply mapped from
$[-1,1]$ to $[0,L]$. The same formula also works for a polynomial curved
1D geometry embedded in 2D or 3D.

In [7]:
# ------------------------------------------------------------------
# Global physical coordinates of the GLL DOFs
# ------------------------------------------------------------------

X_gll_global = fill(NaN, ndof, 3)
assigned = falses(ndof)

for e in axes(connectivity, 1)

    gmsh_tags = vec(connectivity[e, :])
    gll_tags  = gmsh_tags[perm_gll]

    gmsh_dofs = [tagToDof[tag] for tag in gmsh_tags]
    gll_dofs  = [tagToDof[tag] for tag in gll_tags]

    X_e = XYZ[gmsh_dofs, :]

    # Same geometry, evaluated at GLL nodes.
    X_gll_e = R * X_e

    for i in 1:numNodes
        gi = gll_dofs[i]

        if assigned[gi]
            norm(X_gll_global[gi, :] - X_gll_e[i, :]) < 1e-12 ||
                error("Inconsistent GLL coordinate at shared node.")
        else
            X_gll_global[gi, :] .= X_gll_e[i, :]
            assigned[gi] = true
        end
    end
end

@assert all(assigned)

# Display nodes ordered by their spectral physical coordinate.
ix = sortperm(X_gll_global[:, 1])

println("Global Gmsh and GLL coordinates:")
display(
    hcat(
        nodeTags[ix],
        XYZ[ix, 1],
        X_gll_global[ix, 1]
    )
)

Global Gmsh and GLL coordinates:


13×3 Matrix{Float64}:
  1.0  0.0        0.0
  5.0  0.0833333  0.0575577
  6.0  0.166667   0.166667
  7.0  0.25       0.275776
  3.0  0.333333   0.333333
  8.0  0.416667   0.390891
  9.0  0.5        0.5
 10.0  0.583333   0.609109
  4.0  0.666667   0.666667
 11.0  0.75       0.724224
 12.0  0.833333   0.833333
 13.0  0.916667   0.942442
  2.0  1.0        1.0

## 5. Independent reference integration

Before involving the LowLevelFEM weak-form assembly, we verify the basis
transformation independently.

For a scalar 1D model with unit coefficients,

$$
K_{ij}=\int_\Omega \frac{dN_i}{ds}\frac{dN_j}{ds}\,ds,
\qquad
M_{ij}=\int_\Omega N_iN_j\,ds.
$$

A sufficiently accurate Gauss–Legendre rule is used here only as a reference.

In [8]:
"""
    gauss_legendre(n)

Compute the `n`-point Gauss–Legendre rule on `[-1,1]` using the
Golub–Welsch eigenvalue construction.
"""
function gauss_legendre(n::Int)
    n ≥ 1 || throw(ArgumentError("n must be positive."))

    n == 1 && return ([0.0], [2.0])

    β = [k / sqrt(4k^2 - 1) for k in 1:n-1]
    F = eigen(SymTridiagonal(zeros(Float64, n), β))

    ξ = F.values
    w = 2 .* F.vectors[1, :].^2

    return ξ, w
end


"""
    scalar_1d_element_matrices(nodes, Xnodes; nq=length(nodes)+2)

Assemble consistent scalar 1D stiffness and mass matrices using the nodal
basis defined by `nodes` and the isoparametric geometry `Xnodes`.
"""
function scalar_1d_element_matrices(
    nodes::AbstractVector,
    Xnodes::AbstractMatrix;
    nq=length(nodes) + 2
)
    ξq, wq = gauss_legendre(nq)
    N, D = lagrange_matrices(nodes, ξq)

    dX_dξ = D * Xnodes
    J = vec(sqrt.(sum(abs2, dX_dξ; dims=2)))

    all(J .> 0) || error("Degenerate element mapping.")

    nn = length(nodes)
    K = zeros(Float64, nn, nn)
    M = zeros(Float64, nn, nn)

    for q in eachindex(ξq)
        Nq = view(N, q, :)
        Dq = view(D, q, :)

        K .+= (wq[q] / J[q]) .* (Dq * Dq')
        M .+= (wq[q] * J[q]) .* (Nq * Nq')
    end

    return K, M
end


K_gmsh, M_gmsh =
    scalar_1d_element_matrices(ξ_gmsh, X_gmsh; nq=p + 2)

K_gll_direct, M_gll_direct =
    scalar_1d_element_matrices(ξ_gll, X_gll; nq=p + 2)

K_gll_T = T' * K_gmsh * T
M_gll_T = T' * M_gmsh * T

println("Transformation errors:")
@show norm(K_gll_T - K_gll_direct, Inf)
@show norm(M_gll_T - M_gll_direct, Inf)

Transformation errors:
norm(K_gll_T - K_gll_direct, Inf) = 7.815970093361102e-14
norm(M_gll_T - M_gll_direct, Inf) = 5.551115123125783e-16


5.551115123125783e-16

If both errors are close to machine precision, this is the central proof of
concept:

> the GLL element matrix can be obtained from the existing high-order Gmsh
> finite-element matrix by a pure square basis transformation.

No spectral-specific stiffness assembly is required for this step.

## 6. GLL nodal quadrature and the diagonal mass matrix

Now the quadrature points are chosen to be identical to the GLL interpolation
nodes.

Because

$$
N_i(\xi_q)=\delta_{iq},
$$

the mass matrix obtained by GLL nodal quadrature is diagonal:

$$
M_{ij}^{\mathrm{GLL}}
=
\delta_{ij}\,w_i\,J(\xi_i)
$$

for unit density and unit cross-sectional area.

For a straight affine 1D element, GLL quadrature also integrates the stiffness
matrix exactly, because the stiffness integrand has polynomial degree
$2p-2$, while the $(p+1)$-point GLL rule is exact through degree $2p-1$.

In [9]:
"""
    gll_nodal_matrices(nodes, weights, Xnodes)

Assemble the scalar 1D stiffness matrix and the diagonal mass matrix using
GLL nodal quadrature.
"""
function gll_nodal_matrices(
    nodes::AbstractVector,
    weights::AbstractVector,
    Xnodes::AbstractMatrix
)
    N, D = lagrange_matrices(nodes, nodes)

    dX_dξ = D * Xnodes
    J = vec(sqrt.(sum(abs2, dX_dξ; dims=2)))

    all(J .> 0) || error("Degenerate element mapping.")

    nn = length(nodes)
    K = zeros(Float64, nn, nn)

    for q in 1:nn
        Dq = view(D, q, :)
        K .+= (weights[q] / J[q]) .* (Dq * Dq')
    end

    Mdiag = weights .* J
    M = Matrix(Diagonal(Mdiag))

    return K, M, J
end


K_gll_nodal, M_gll_nodal, J_gll =
    gll_nodal_matrices(ξ_gll, w_gll, X_gll)

println("GLL nodal quadrature checks:")
@show norm(K_gll_nodal - K_gll_direct, Inf)

offdiag =
    M_gll_nodal - Matrix(Diagonal(diag(M_gll_nodal)))

@show norm(offdiag, Inf)
@show sum(diag(M_gll_nodal))
@show L

GLL nodal quadrature checks:
norm(K_gll_nodal - K_gll_direct, Inf) = 5.755396159656812e-13
norm(offdiag, Inf) = 0.0
sum(diag(M_gll_nodal)) = 0.3333333333333333
L = 1.0


1.0

In [10]:
"""
    append_element!(I, J, V, dofs, A)

Append a local element matrix to global sparse IJV storage.
"""
function append_element!(
    I::Vector{Int},
    J::Vector{Int},
    V::Vector{Float64},
    dofs,
    A
)
    for j in axes(A, 2)
        for i in axes(A, 1)
            value = A[i, j]

            if value != 0.0
                push!(I, dofs[i])
                push!(J, dofs[j])
                push!(V, value)
            end
        end
    end

    return
end


# Global IJV storage
IK = Int[]; JK = Int[]; VK = Float64[]
IM = Int[]; JM = Int[]; VM = Float64[]

IKg = Int[]; JKg = Int[]; VKg = Float64[]
IMg = Int[]; JMg = Int[]; VMg = Float64[]

IKn = Int[]; JKn = Int[]; VKn = Float64[]
IMn = Int[]; JMn = Int[]; VMn = Float64[]


for e in axes(connectivity, 1)

    gmsh_tags = vec(connectivity[e, :])
    gll_tags  = gmsh_tags[perm_gll]

    gmsh_dofs = [tagToDof[tag] for tag in gmsh_tags]
    gll_dofs  = [tagToDof[tag] for tag in gll_tags]

    X_e = XYZ[gmsh_dofs, :]
    X_gll_e = R * X_e

    # Ordinary Gmsh basis
    Ke, Me =
        scalar_1d_element_matrices(
            ξ_gmsh,
            X_e;
            nq=p + 2
        )

    # Consistent GLL basis — independent reference
    Kg, Mg =
        scalar_1d_element_matrices(
            ξ_gll,
            X_gll_e;
            nq=p + 2
        )

    # GLL nodal quadrature
    Kn, Mn, _ =
        gll_nodal_matrices(
            ξ_gll,
            w_gll,
            X_gll_e
        )

    append_element!(IK,  JK,  VK,  gmsh_dofs, Ke)
    append_element!(IM,  JM,  VM,  gmsh_dofs, Me)

    append_element!(IKg, JKg, VKg, gll_dofs, Kg)
    append_element!(IMg, JMg, VMg, gll_dofs, Mg)

    append_element!(IKn, JKn, VKn, gll_dofs, Kn)
    append_element!(IMn, JMn, VMn, gll_dofs, Mn)
end


K_gmsh_global =
    sparse(IK, JK, VK, ndof, ndof)

M_gmsh_global =
    sparse(IM, JM, VM, ndof, ndof)

K_gll_direct_global =
    sparse(IKg, JKg, VKg, ndof, ndof)

M_gll_direct_global =
    sparse(IMg, JMg, VMg, ndof, ndof)

K_gll_nodal_global =
    sparse(IKn, JKn, VKn, ndof, ndof)

M_gll_nodal_global =
    sparse(IMn, JMn, VMn, ndof, ndof)

13×13 SparseMatrixCSC{Float64, Int64} with 13 stored entries:
 0.0166667   ⋅          ⋅         …   ⋅          ⋅         ⋅ 
  ⋅         0.0166667   ⋅             ⋅          ⋅         ⋅ 
  ⋅          ⋅         0.0333333      ⋅          ⋅         ⋅ 
  ⋅          ⋅          ⋅             ⋅          ⋅         ⋅ 
  ⋅          ⋅          ⋅             ⋅          ⋅         ⋅ 
  ⋅          ⋅          ⋅         …   ⋅          ⋅         ⋅ 
  ⋅          ⋅          ⋅             ⋅          ⋅         ⋅ 
  ⋅          ⋅          ⋅             ⋅          ⋅         ⋅ 
  ⋅          ⋅          ⋅             ⋅          ⋅         ⋅ 
  ⋅          ⋅          ⋅             ⋅          ⋅         ⋅ 
  ⋅          ⋅          ⋅         …  0.0907407   ⋅         ⋅ 
  ⋅          ⋅          ⋅             ⋅         0.118519   ⋅ 
  ⋅          ⋅          ⋅             ⋅          ⋅        0.0907407

## 7. Connection to the existing `consistentToLumped`

For this straight, constant-coefficient 1D test, the row sums of the
**consistent GLL-basis mass matrix** should coincide with the diagonal entries
obtained from GLL nodal quadrature.

This is especially interesting for LowLevelFEM because a row-sum lumping
utility already exists.

In [11]:
m_rowsum = vec(sum(M_gll_direct, dims=2))
m_gll = diag(M_gll_nodal)

println("row-sum consistent mass vs. GLL nodal mass:")
display(hcat(m_rowsum, m_gll, m_rowsum - m_gll))

@show norm(m_rowsum - m_gll, Inf)

row-sum consistent mass vs. GLL nodal mass:


5×3 Matrix{Float64}:
 0.0166667  0.0166667  -3.01842e-16
 0.0907407  0.0907407   4.16334e-16
 0.118519   0.118519   -3.05311e-16
 0.0907407  0.0907407   7.07767e-16
 0.0166667  0.0166667  -2.77556e-17

norm(m_rowsum - m_gll, Inf) = 7.077671781985373e-16


7.077671781985373e-16

This equivalence is very encouraging for a minimal-intrusion implementation:

1. assemble the ordinary consistent matrix with the existing LowLevelFEM code,
2. transform it to the GLL basis,
3. apply the existing row-sum lumping operation.

However, this notebook tests only the simple affine/constant-coefficient case.
For curved mappings, spatially varying density/material data, and higher
dimensions, the relation between exact consistent row-sum lumping and direct
GLL nodal quadrature must be checked separately.

## 8. What this prototype should establish

If all checks above pass, we have verified the most important architectural
idea:

$$
\boxed{
\text{existing Gmsh/LLFEM element}
\;\xrightarrow{\;T\;}
\text{GLL spectral representation}
}
$$

with

$$
\boxed{
K_G=T^\mathsf{T}K_ET,
\qquad
M_G=T^\mathsf{T}M_ET.
}
$$

The next experiment should use **several 1D elements** and assemble a sparse
global transformation matrix $T$. That will show whether the same idea can be
inserted into the existing LowLevelFEM transformation pipeline as cleanly as
the current P1/P2 reduction.

Only after that is stable does it make sense to move to tensor-product
quadrilateral/hexahedral elements.

In [12]:
K_gll_T_global =
    T_global' * K_gmsh_global * T_global

M_gll_T_global =
    T_global' * M_gmsh_global * T_global


println("Global transformation errors:")

@show norm(
    Matrix(K_gll_T_global - K_gll_direct_global),
    Inf
)

@show norm(
    Matrix(M_gll_T_global - M_gll_direct_global),
    Inf
)


println()
println("GLL nodal quadrature:")

@show norm(
    Matrix(K_gll_nodal_global - K_gll_direct_global),
    Inf
)

Moff =
    M_gll_nodal_global -
    spdiagm(0 => diag(M_gll_nodal_global))

@show norm(Matrix(Moff), Inf)

m_rowsum =
    vec(sum(M_gll_direct_global, dims=2))

m_gll =
    diag(M_gll_nodal_global)

@show norm(m_rowsum - m_gll, Inf)

@show sum(m_gll)
@show L

Global transformation errors:
norm(Matrix(K_gll_T_global - K_gll_direct_global), Inf) = 3.979039320256561e-13
norm(Matrix(M_gll_T_global - M_gll_direct_global), Inf) = 1.8457457784393227e-15

GLL nodal quadrature:
norm(Matrix(K_gll_nodal_global - K_gll_direct_global), Inf) = 5.755396159656812e-13
norm(Matrix(Moff), Inf) = 0.0
norm(m_rowsum - m_gll, Inf) = 1.887379141862766e-15
sum(m_gll) = 0.9999999999999998
L = 1.0


1.0